In [ ]:
#导入相关库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import rcParams

In [ ]:
#设置中文字体
import matplotlib
matplotlib.rc("font", family = 'Microsoft Yahei')

In [ ]:
# 读取后续分析需要的字段，控制内存占用
orders = pd.read_csv(
    'Dataset_Customer/orders.csv',
    usecols=[
        'order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
        'order_delivered_customer_date', 'order_estimated_delivery_date'
    ]
)
customers = pd.read_csv(
    'Dataset_Customer/customers.csv',
    usecols=['customer_id', 'customer_unique_id', 'customer_city', 'customer_state', 'customer_segment']
)
payments = pd.read_csv(
    'Dataset_Customer/order_payments.csv',
    usecols=['order_id', 'payment_sequential', 'payment_type', 'payment_value']
)
print('数据读取成功')
for name, data in {
    'orders': orders, 'customers': customers, 'payments': payments
}.items():
    print(f'{name}: {data.shape}')


In [ ]:
# 构建已完成订单的一单一行核心行为表
# 1. 仅保留已完成订单，避免取消订单的支付金额污染 GMV、LTV 与 AOV
valid_order_status = 'delivered'
valid_orders = orders.loc[
    orders['order_status'].eq(valid_order_status),
    ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp']
].copy()
print(f"保留已完成订单：{len(valid_orders):,} 笔；筛除取消订单：{len(orders) - len(valid_orders):,} 笔")

# 2. 支付明细先聚合至订单粒度，确保每个 order_id 只有一行
# payment_value 为同一订单所有支付记录之和，适用于订单级 GMV、LTV 和 AOV 计算
# 不按 payment_value 去重：相同金额但不同 payment_sequential 的分笔支付均为有效支付。
payment_by_order = (
    payments.groupby('order_id', as_index=False)
    .agg(
        payment_value=('payment_value', 'sum'),
        payment_record_count=('payment_sequential', 'count')
    )
)
print(f"支付记录：{len(payments):,} 行；聚合后订单支付记录：{len(payment_by_order):,} 行")
print(f"存在多笔支付的订单数：{(payment_by_order['payment_record_count'] > 1).sum():,} 笔")

# 3. 使用一对一关联校验，防止支付明细或客户表的重复记录放大订单金额
order_level_df = valid_orders.merge(
    customers,
    on='customer_id',
    how='left',
    validate='one_to_one'
).merge(
    payment_by_order,
    on='order_id',
    how='left',
    validate='one_to_one'
)

# 每行代表一笔已完成订单，后续用户和金额指标均基于统一订单粒度
user_behavior_df = order_level_df[
    ['customer_unique_id', 'order_id', 'order_purchase_timestamp', 'payment_value']
].copy()

print("核心行为表（已完成订单，一单一行）")
print(user_behavior_df.head())
print("核心行为表形状：", user_behavior_df.shape)
print("去重订单数：", user_behavior_df['order_id'].nunique())
# 后续指标只使用 user_behavior_df；释放关联中间表以降低内存占用。
del valid_orders, payment_by_order, order_level_df


In [ ]:
# 缺失值处理：仅检查订单级核心字段
required_columns = [
    'customer_unique_id',
    'order_id',
    'order_purchase_timestamp',
    'payment_value'
]
print("缺失值个数：")
print(user_behavior_df[required_columns].isnull().sum())
user_behavior_df = user_behavior_df.dropna(subset=required_columns).copy()
print("去除缺失值后：")
print(user_behavior_df[required_columns].isnull().sum())


In [ ]:
# 订单粒度校验：核心行为表必须一单一行，不再按支付明细行去重
duplicate_order_count = user_behavior_df['order_id'].duplicated().sum()
print(f"重复订单数：{duplicate_order_count}")
assert duplicate_order_count == 0, "发现重复订单，请检查订单与支付表的关联粒度。"
print("订单粒度校验通过：每个 order_id 仅保留一行。")


In [ ]:
# 时间特征处理
# 将时间戳转换成日期格式
user_behavior_df['order_purchase_timestamp'] = pd.to_datetime(user_behavior_df['order_purchase_timestamp'])

# 提取相关的时间特征
# 日
user_behavior_df['order_date'] = user_behavior_df['order_purchase_timestamp'].dt.date

# 周
user_behavior_df['order_week'] = user_behavior_df['order_purchase_timestamp'].dt.to_period('W')

# 月
user_behavior_df['order_month'] = user_behavior_df['order_purchase_timestamp'].dt.to_period('M')

# 查看数据合并并清洗后的字段名
print("数据合并并清洗后的字段名")
print(user_behavior_df.columns)

In [ ]:
# DAU
# 计算DAU，并按日期排序（关键一步）
DAU = (user_behavior_df.groupby('order_date')['customer_unique_id']
       .nunique()
       .sort_index())

# DAU可视化
plt.figure(figsize=(14, 6))
plt.plot(DAU.index, DAU.values)   # 明确用折线
plt.title('Daily Active Users')
plt.xlabel('Date')
plt.ylabel('DAU')
plt.show()

In [ ]:
#WAU
#计算WAU
WAU = (user_behavior_df.groupby('order_week')['customer_unique_id']
      .nunique())

#WAU可视化
plt.figure(figsize=(14,6))
WAU.plot()
plt.title('Weekly Active Users')
plt.xlabel('Week')
plt.ylabel('WAU')
plt.show()

In [ ]:
# MAU
MAU = user_behavior_df.groupby('order_month')['customer_unique_id'].nunique()
plt.figure(figsize=(14, 6))
MAU.plot()
plt.title('Monthly Active Users')
plt.xlabel('Month')
plt.ylabel('MAU')
plt.show()


In [ ]:
#计算GMV
GMV = (user_behavior_df.groupby('order_date')['payment_value']
      .sum())

#GMV可视化
plt.figure(figsize=(14,6))
GMV.plot()
plt.title('Daily GMV')
plt.xlabel('Date')
plt.ylabel('GMV')
plt.show()

In [ ]:
# 用户留存分析 按"首次购买周"分组
# 获取用户首次购买时间
first_purchase_week = (user_behavior_df.groupby('customer_unique_id')['order_purchase_timestamp']
                       .min()
                       .reset_index())

# 提取 Cohort_week（首次购买周）
first_purchase_week['cohort_week'] = (first_purchase_week['order_purchase_timestamp']
                                      .dt.to_period('W'))

# 合并回原表
cohort_week_df = user_behavior_df.merge(
    first_purchase_week[['customer_unique_id', 'cohort_week']],
    on='customer_unique_id',
    how='left')

# 提取 week_index（距首次购买过去了几周）
cohort_week_df['week_index'] = ((cohort_week_df['order_week'].dt.start_time -
                                 cohort_week_df['cohort_week'].dt.start_time)
                                .dt.days // 7)

# 统计每周留存的用户数
cohort_data = (cohort_week_df.groupby(['cohort_week', 'week_index'])['customer_unique_id']
               .nunique()
               .reset_index())

# 构建 cohort 矩阵
cohort_matrix = cohort_data.pivot(index='cohort_week',
                                  columns='week_index',
                                  values='customer_unique_id')

# 计算留存率
cohort_size = cohort_matrix.iloc[:, 0]
retention_matrix = cohort_matrix.divide(cohort_size, axis=0)

# 只看前12周（避免矩阵过宽、过于稀疏、有大量NaN值）
retention_matrix = retention_matrix.iloc[:, :12]

# 删除最前面异常cohort（数据集起始阶段可能存在用户历史行为缺失）
retention_matrix = retention_matrix.iloc[2:]

# 绘制热力图
retention_plot = retention_matrix.tail(15)
plt.figure(figsize=(14, 8))
sns.heatmap(retention_plot, annot=True, fmt='.1%', cmap='Blues', vmin=0, vmax=0.3)
plt.title('Weekly Cohort Retention', fontsize=18)
plt.xlabel('Weeks Since First Purchase')
plt.ylabel('Cohort Week')
plt.show()

In [ ]:
#用户复购分析
#统计每个用户订单数
user_order_count = (user_behavior_df.groupby('customer_unique_id')['order_id']
                   .nunique()
                   .reset_index())
user_order_count.columns = ['customer_unique_id', 'order_count']

#统计购买次数分布
purchase_distribution = (user_order_count['order_count']
                        .value_counts()
                        .sort_index())
print('购买次数分别是')
print(purchase_distribution)

#计算购买次数占比
purchase_ratio = (purchase_distribution / purchase_distribution.sum())
print('购买次数的占比是')
print(purchase_ratio)

#占比可视化
plt.figure(figsize=(10,6))
purchase_distribution[:10].plot(kind='bar')
plt.title('Purchase Frequency Distribution')
plt.xlabel('Purchase Count')
plt.ylabel('Number of Users')
plt.show()

In [ ]:
# 用户流失分析：先找出每位用户最近一次购买时间
# 每位用户只保留其最后一次已完成订单的下单时间。
last_purchase = (
    user_behavior_df.groupby('customer_unique_id')['order_purchase_timestamp']
    .max()
    .reset_index(name='last_purchase_at')
)

# 快照日设为数据中最后一次下单的次日。
snapshot_date = user_behavior_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
last_purchase['days_since_last_purchase'] = (
    snapshot_date - last_purchase['last_purchase_at']
).dt.days

print(f'数据快照日：{snapshot_date.date()}')
display(last_purchase.head())


In [ ]:
# 定义流失：超过 90 天未购买的用户视为流失用户
CHURN_THRESHOLD_DAYS = 90
last_purchase['is_churn'] = (
    last_purchase['days_since_last_purchase'] > CHURN_THRESHOLD_DAYS
)

churn_rate = last_purchase['is_churn'].mean()
churn_user_count = last_purchase['is_churn'].sum()
retained_user_count = (~last_purchase['is_churn']).sum()

print(f'流失判定规则：超过 {CHURN_THRESHOLD_DAYS} 天未购买')
print(f'流失用户数：{churn_user_count:,} 人')
print(f'未流失用户数：{retained_user_count:,} 人')
print(f'用户流失率：{churn_rate:.2%}')

plt.figure(figsize=(6, 5))
last_purchase['is_churn'].value_counts().rename({False: '未流失', True: '已流失'}).plot(kind='bar')
plt.title('Churn User Distribution')
plt.xlabel('User Status')
plt.ylabel('User Count')
plt.xticks(rotation=0)
plt.show()


In [ ]:
# 计算 AOV（用户累计已完成订单支付金额 / 用户已完成订单数）
user_analysis = (
    user_behavior_df.groupby('customer_unique_id')
    .agg(order_count=('order_id', 'nunique'), total_spent=('payment_value', 'sum'))
    .reset_index()
)
user_analysis['AOV'] = user_analysis['total_spent'] / user_analysis['order_count']
user_analysis.head()


In [ ]:
# 合并流失标签与用户价值指标，比较流失与未流失用户
churn_analysis = last_purchase.merge(
    user_analysis,
    on='customer_unique_id',
    how='left',
    validate='one_to_one'
)
churn_comparison = churn_analysis.groupby('is_churn').agg(
    user_count=('customer_unique_id', 'size'),
    avg_order_count=('order_count', 'mean'),
    avg_total_spent=('total_spent', 'mean'),
    avg_AOV=('AOV', 'mean')
)
display(churn_comparison)


In [ ]:
#用户生命周期价值分析
#查看分布
user_analysis['total_spent'].describe()

In [ ]:
# LTV分布图
plt.figure(figsize=(10, 6))
sns.histplot(user_analysis['total_spent'], bins=100, kde=True)  # kde=True 保留那条拟合曲线
plt.xlim(0, 20000)
plt.title('User LTV Distribution')
plt.xlabel('Total Spending')
plt.ylabel('User Count')
plt.show()

In [ ]:
# 定义高价值用户（累计消费前 10%）
threshold = user_analysis['total_spent'].quantile(0.9)
user_analysis['high_value_user'] = user_analysis['total_spent'] >= threshold
GMV_contribution = user_analysis.groupby('high_value_user')['total_spent'].sum()
GMV_ratio = GMV_contribution / GMV_contribution.sum()
print(f'高价值用户门槛：{threshold:.2f}')
print('高价值用户 GMV 贡献：')
print(GMV_ratio)


In [ ]:
# 商品类目与预估商品毛利分析
# 预估商品毛利 = 含折后商品收入 - 商品成本；不计运费、履约和营销费用。
items = pd.read_csv(
    'Dataset_Customer/order_items.csv',
    usecols=['order_id', 'product_id', 'price', 'freight_value', 'discount_rate']
)
products = pd.read_csv(
    'Dataset_Customer/products.csv',
    usecols=['product_id', 'product_category_name', 'cost']
)
item_financial_df = (
    user_behavior_df[['order_id']]
    .merge(items, on='order_id', how='inner', validate='one_to_many')
    .merge(products, on='product_id', how='left', validate='many_to_one')
)
item_financial_df['product_category_name'] = item_financial_df['product_category_name'].fillna('Unknown')
item_financial_df['net_item_revenue'] = item_financial_df['price'] * (1 - item_financial_df['discount_rate'])
item_financial_df['estimated_gross_profit'] = item_financial_df['net_item_revenue'] - item_financial_df['cost']
category_kpi = (
    item_financial_df.groupby('product_category_name')
    .agg(
        item_count=('order_id', 'size'),
        order_count=('order_id', 'nunique'),
        net_item_revenue=('net_item_revenue', 'sum'),
        product_cost=('cost', 'sum'),
        estimated_gross_profit=('estimated_gross_profit', 'sum')
    )
    .reset_index()
)
category_kpi['estimated_gross_margin'] = (
    category_kpi['estimated_gross_profit'] / category_kpi['net_item_revenue']
)
category_kpi = category_kpi.sort_values('net_item_revenue', ascending=False)
display(category_kpi.head(15))
# 地域分析只需订单级收入，聚合后释放商品明细以控制内存。
order_revenue_df = item_financial_df.groupby('order_id', as_index=False).agg(
    net_item_revenue=('net_item_revenue', 'sum')
)
del item_financial_df, items, products


In [ ]:
# 地域分析：按客户所在州汇总已完成订单和商品收入
order_region_df = (
    orders.loc[
        orders['order_status'].eq(valid_order_status),
        ['order_id', 'customer_id']
    ]
    .merge(
        customers[['customer_id', 'customer_unique_id', 'customer_state', 'customer_city']],
        on='customer_id', how='left', validate='one_to_one'
    )
    .merge(order_revenue_df, on='order_id', how='left', validate='one_to_one')
)
regional_kpi = (
    order_region_df.groupby('customer_state')
    .agg(
        user_count=('customer_unique_id', 'nunique'),
        order_count=('order_id', 'nunique'),
        net_item_revenue=('net_item_revenue', 'sum')
    )
    .sort_values('net_item_revenue', ascending=False)
)
display(regional_kpi.head(10))
del order_region_df, order_revenue_df


In [ ]:
# 物流时效分析：仅使用有签收时间和预计送达时间的已完成订单
delivery_df = orders.loc[
    orders['order_status'].eq(valid_order_status),
    ['order_id', 'order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']
].copy()
for column in ['order_purchase_timestamp', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
    delivery_df[column] = pd.to_datetime(delivery_df[column])
delivery_df = delivery_df.dropna(subset=['order_delivered_customer_date', 'order_estimated_delivery_date'])
delivery_df['delivery_days'] = (
    delivery_df['order_delivered_customer_date'] - delivery_df['order_purchase_timestamp']
).dt.total_seconds() / 86400
delivery_df['late_days'] = (
    delivery_df['order_delivered_customer_date'] - delivery_df['order_estimated_delivery_date']
).dt.total_seconds() / 86400
late_delivery_rate = (delivery_df['late_days'] > 0).mean()
print(f'延迟送达率：{late_delivery_rate:.2%}')
display(delivery_df[['delivery_days', 'late_days']].describe(percentiles=[.5, .75, .9, .95]))


In [ ]:
# 评价分析：先聚合到订单粒度，避免一笔订单存在多条评价时重复计算
reviews = pd.read_csv(
    'Dataset_Customer/order_reviews.csv',
    usecols=['order_id', 'review_score']
)
review_by_order = (
    reviews.groupby('order_id', as_index=False)
    .agg(review_score=('review_score', 'mean'), review_record_count=('review_score', 'size'))
)
order_review_df = user_behavior_df[['order_id']].merge(
    review_by_order, on='order_id', how='left', validate='one_to_one'
)
review_coverage = order_review_df['review_score'].notna().mean()
review_distribution = order_review_df['review_score'].value_counts(normalize=True).sort_index()
print(f'已完成订单评价覆盖率：{review_coverage:.2%}')
print('评价得分占比：')
print(review_distribution)
del reviews, review_by_order, order_review_df


In [ ]:
# 支付偏好分析：支付方式不是获客渠道，不能用于渠道归因
payment_type_kpi = (
    payments.groupby('payment_type')
    .agg(payment_record_count=('order_id', 'size'), payment_value=('payment_value', 'sum'))
    .sort_values('payment_value', ascending=False)
)
payment_type_kpi['payment_value_share'] = payment_type_kpi['payment_value'] / payment_type_kpi['payment_value'].sum()
display(payment_type_kpi)
print('数据集中未提供 channel、source 或 campaign 字段，因此不能开展获客渠道效果或渠道 ROI 分析。')
